In [1]:
# =========================
# Face crop + Liveness test
# =========================

import os
import math
import random
from pathlib import Path

import cv2
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image

from insightface.app import FaceAnalysis

# -------------------------------------------------
# 1) CONFIG
# -------------------------------------------------
VIDEO_PATH = r"C:\Users\super\Videos\2026-03-07 01-14-12.mkv"  # <-- change this
LIVENESS_REPO_DIR = r"C:\DSP\face_liveness_vit"  # <-- folder created by snapshot_download
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Sample every Nth frame for liveness
FRAME_STRIDE = 6          # lower = more frames checked
MAX_SAMPLED_FACES = 12    # max aligned face crops to evaluate
DET_SIZE = (640, 640)
IMG_SIZE = 224

# Liveness threshold from model card:
# spoof_prob < 0.1199 => live
LIVENESS_SPOOF_THRESHOLD = 0.1199

# -------------------------------------------------


In [2]:
# -------------------------------------------------
# 2) CUSTOM LIVENESS ViT CLASS
#    Adjusted to match checkpoint key structure:
#    encoder.transformer_encoder.layers...
# -------------------------------------------------
class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=128):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size * self.grid_size
        self.proj = nn.Conv2d(
            in_chans, embed_dim,
            kernel_size=patch_size,
            stride=patch_size
        )

    def forward(self, x):
        # x: [B, 3, 224, 224]
        x = self.proj(x)                  # [B, D, 14, 14]
        x = x.flatten(2).transpose(1, 2) # [B, 196, D]
        return x


class Encoder(nn.Module):
    def __init__(self, d_model=128, nhead=4, num_layers=2, mlp_ratio=4.0, dropout=0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=int(d_model * mlp_ratio),
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

    def forward(self, x):
        return self.transformer_encoder(x)


class LivenessViT(nn.Module):
    def __init__(
        self,
        img_size=224,
        patch_size=16,
        d_model=128,
        nhead=4,
        num_layers=2,
        num_classes=2,
        mlp_ratio=4.0,
        dropout=0.1,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(
            img_size=img_size,
            patch_size=patch_size,
            in_chans=3,
            embed_dim=d_model
        )
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.zeros(1, 1 + num_patches, d_model))
        self.pos_drop = nn.Dropout(dropout)

        self.encoder = Encoder(
            d_model=d_model,
            nhead=nhead,
            num_layers=num_layers,
            mlp_ratio=mlp_ratio,
            dropout=dropout,
        )

        self.head = nn.Linear(d_model, num_classes)

        self._init_weights()

    def _init_weights(self):
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.head.weight, std=0.02)
        nn.init.zeros_(self.head.bias)

    def forward(self, x):
        x = self.patch_embed(x)                 # [B, N, D]
        B = x.shape[0]
        cls = self.cls_token.expand(B, -1, -1) # [B, 1, D]
        x = torch.cat([cls, x], dim=1)         # [B, 1+N, D]
        x = x + self.pos_embed[:, :x.size(1), :]
        x = self.pos_drop(x)
        x = self.encoder(x)
        logits = self.head(x[:, 0])            # CLS token
        return logits


In [3]:
# -------------------------------------------------
# 3) LOAD LIVENESS MODEL
# -------------------------------------------------
checkpoint_path = os.path.join(LIVENESS_REPO_DIR, "model.pt")
if not os.path.exists(checkpoint_path):
    raise FileNotFoundError(f"Could not find liveness checkpoint at: {checkpoint_path}")

liveness_model = LivenessViT(
    img_size=224,
    patch_size=16,
    d_model=128,
    nhead=4,
    num_layers=2,
    num_classes=2,
).to(DEVICE)

checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)

# Model card suggests checkpoint['model']
state_dict = checkpoint["model"] if isinstance(checkpoint, dict) and "model" in checkpoint else checkpoint
liveness_model.load_state_dict(state_dict, strict=True)
liveness_model.eval()

liveness_transform = transforms.Compose([
    transforms.Resize(int(IMG_SIZE * 1.14)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
])

print(f"Liveness model loaded on: {DEVICE}")


# -------------------------------------------------


Liveness model loaded on: cuda


c:\Users\super\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [4]:
# 4) LOAD INSIGHTFACE buffalo_l
# -------------------------------------------------
providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if DEVICE == "cuda" else ['CPUExecutionProvider']

face_app = FaceAnalysis(name="buffalo_l", providers=providers)
face_app.prepare(ctx_id=0 if DEVICE == "cuda" else -1, det_size=DET_SIZE)

print("InsightFace buffalo_l loaded.")


# -------------------------------------------------


c:\Users\super\AppData\Local\Programs\Python\Python312\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\super/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\super/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\super/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\super/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\super/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.

In [5]:
# 5) FACE ALIGNMENT / CROP HELPER
#    We use the aligned crop if available from face.kps via norm_crop,
#    otherwise a bbox crop fallback.
# -------------------------------------------------
from insightface.utils import face_align

def get_aligned_face_224(frame_bgr, face_obj, out_size=224):
    """
    Returns aligned RGB face crop as PIL.Image
    """
    # Preferred: align using keypoints
    if hasattr(face_obj, "kps") and face_obj.kps is not None:
        aligned_bgr = face_align.norm_crop(frame_bgr, landmark=face_obj.kps, image_size=out_size)
        aligned_rgb = cv2.cvtColor(aligned_bgr, cv2.COLOR_BGR2RGB)
        return Image.fromarray(aligned_rgb)

    # Fallback: bbox crop
    x1, y1, x2, y2 = face_obj.bbox.astype(int)
    h, w = frame_bgr.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    crop_bgr = frame_bgr[y1:y2, x1:x2]
    if crop_bgr.size == 0:
        return None
    crop_bgr = cv2.resize(crop_bgr, (out_size, out_size))
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(crop_rgb)


# -------------------------------------------------


In [6]:
# 6) RUN VIDEO -> DETECT FACE -> CROP -> LIVENESS
# -------------------------------------------------
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Could not open video: {VIDEO_PATH}")

sampled_faces = []
frame_idx = 0

while True:
    ok, frame_bgr = cap.read()
    if not ok:
        break

    if frame_idx % FRAME_STRIDE != 0:
        frame_idx += 1
        continue

    faces = face_app.get(frame_bgr)

    if len(faces) > 0:
        # Pick the largest face
        faces = sorted(faces, key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]), reverse=True)
        face = faces[0]

        aligned_pil = get_aligned_face_224(frame_bgr, face, out_size=224)
        if aligned_pil is not None:
            sampled_faces.append(aligned_pil)

    if len(sampled_faces) >= MAX_SAMPLED_FACES:
        break

    frame_idx += 1

cap.release()

if len(sampled_faces) == 0:
    raise RuntimeError("No faces were detected in the sampled video frames.")

print(f"Collected {len(sampled_faces)} aligned face crops.")


# -------------------------------------------------


c:\Users\super\AppData\Local\Programs\Python\Python312\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


Collected 12 aligned face crops.


In [7]:
# -------------------------------------------
# DEBUG CELL: Save Cropped Face Frames
# Uses the already-loaded video path and face detector.
# Saves aligned 224x224 face crops into ./Crops.
# -------------------------------------------
import os
import cv2
from datetime import datetime

crop_dir = os.path.join(os.getcwd(), "Crops")
os.makedirs(crop_dir, exist_ok=True)

cap = cv2.VideoCapture(VIDEO_PATH)
frame_id = 0
saved = 0
run_stamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("Saving 224x224 crops to:", crop_dir)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    faces = face_app.get(frame)

    for i, face in enumerate(faces):
        aligned_pil = get_aligned_face_224(frame, face, out_size=224)
        if aligned_pil is None:
            continue

        filename = f"{run_stamp}_frame{frame_id}_face{i}.jpg"
        path = os.path.join(crop_dir, filename)
        aligned_pil.save(path, quality=95)
        saved += 1

    frame_id += 1

cap.release()
print(f"Done. Saved {saved} aligned 224x224 face crops.")

Saving 224x224 crops to: c:\DSP\Crops
Done. Saved 176 aligned 224x224 face crops.


In [8]:
# 7) LIVENESS INFERENCE ON ALL SAMPLED CROPS
# -------------------------------------------------
all_spoof_probs = []
all_live_flags = []

with torch.no_grad():
    for i, face_pil in enumerate(sampled_faces):
        x = liveness_transform(face_pil).unsqueeze(0).to(DEVICE)
        logits = liveness_model(x)
        probs = torch.softmax(logits, dim=1)

        # As per model card: class 1 = spoof
        spoof_prob = probs[0, 1].item()
        is_live = spoof_prob < LIVENESS_SPOOF_THRESHOLD

        all_spoof_probs.append(spoof_prob)
        all_live_flags.append(is_live)

        print(f"FrameFace {i:02d}: spoof_prob={spoof_prob:.4f} | live={is_live}")

mean_spoof = sum(all_spoof_probs) / len(all_spoof_probs)
live_ratio = sum(all_live_flags) / len(all_live_flags)

# Simple session-level decision
session_live = (mean_spoof < LIVENESS_SPOOF_THRESHOLD) and (live_ratio >= 0.6)

print("\n===== SESSION SUMMARY =====")
print(f"Mean spoof probability : {mean_spoof:.4f}")
print(f"Live frame ratio       : {live_ratio:.2%}")
print(f"Session considered live: {session_live}")

FrameFace 00: spoof_prob=0.1669 | live=False
FrameFace 01: spoof_prob=0.1401 | live=False
FrameFace 02: spoof_prob=0.1341 | live=False
FrameFace 03: spoof_prob=0.1283 | live=False
FrameFace 04: spoof_prob=0.1382 | live=False
FrameFace 05: spoof_prob=0.1401 | live=False
FrameFace 06: spoof_prob=0.1271 | live=False
FrameFace 07: spoof_prob=0.1681 | live=False
FrameFace 08: spoof_prob=0.1297 | live=False
FrameFace 09: spoof_prob=0.1478 | live=False
FrameFace 10: spoof_prob=0.1544 | live=False
FrameFace 11: spoof_prob=0.1497 | live=False

===== SESSION SUMMARY =====
Mean spoof probability : 0.1437
Live frame ratio       : 0.00%
Session considered live: False
